In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate

# 1. Cargar el dataset limpio y con features
df = pd.read_csv('../data/processed/consumo_granada_modelo.csv')

# 2. Separar Features (X) y Target (y)
X = df.drop('consumption_kwh', axis=1)
y = df['consumption_kwh']

# 3. División Train (75%) / Test (25%)
# random_state=42 para que siempre salga la misma división (reproducibilidad)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Datos de Entrenamiento: {X_train.shape}")
print(f"Datos de Test (Reservados): {X_test.shape}")

Datos de Entrenamiento: (1460595, 33)
Datos de Test (Reservados): (486866, 33)


In [2]:
# 1. Definir el modelo (Estrategia: Predecir siempre la media)
dummy_model = DummyRegressor(strategy='mean')

# 2. Configurar métricas (Scikit-learn usa "negativo" para errores, luego lo corregimos)
scoring = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error'
}

# 3. Entrenar con 5-Fold CV (Solo en X_train)
print("Entrenando Dummy Regressor con 5-Fold CV...")
cv_results = cross_validate(dummy_model, X_train, y_train, cv=5, scoring=scoring)

# 4. Procesar resultados (invertir signo negativo)
dummy_mae = -cv_results['test_MAE'].mean()
dummy_rmse = -cv_results['test_RMSE'].mean()

# 5. Entrenar el modelo final en todo X_train
dummy_model.fit(X_train, y_train)

print("\nResultados del DUMMY (Baseline):")
print(f"MAE Promedio:  {dummy_mae:.2f} kWh")
print(f"RMSE Promedio: {dummy_rmse:.2f} kWh")

Entrenando Dummy Regressor con 5-Fold CV...

Resultados del DUMMY (Baseline):
MAE Promedio:  1124.49 kWh
RMSE Promedio: 1471.58 kWh


In [3]:
# 6. Exportar el modelo entrenado
joblib.dump(dummy_model, "../data/models/dummy.pkl")
print("\nModelo guardado en: ../data/models/dummy.pkl")

FileNotFoundError: [Errno 2] No such file or directory: '../data/models/dummy.pkl'